In [5]:
import pandas as pd
from collections import defaultdict

In [2]:
df_insta = pd.read_csv('insta_ad_library.csv')

In [3]:
df_insta

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,provider_data.user.id,provider_data.waist,provider_data.rawlog_id,advertisement.details.advert_id,advertiser.details.advertiser_id,advertiser.details.advertiser_real_name,advertiser.details.advertiser_internal_name,...,month,year,country,subfolder,english_waist,targeting_categories_types_german,english_waist_converted,targeting_categories_english,targeting_categories_granularity,selected_ad_library
0,0,0,75883,133579,"[""Der aktuelle Wohnort, den du in deinem Faceb...",155271982,6.641693e+12,3.203010e+09,unicef_deutschland,UNICEF Deutschland,...,9,2024,2024,month=9,"[""The current place of residence you indicated...","['user_characteristic_country', 'user_connecti...",['The current place of residence you indicated...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you ind...,"['18-65+ years old', 'All', ['Germany']]"
1,1,1,56577,68022,"['Der aktuelle Wohnort, den du in deinem Faceb...",155148521,1.202086e+17,1.530361e+09,lightroom,Adobe Photoshop Lightroom,...,9,2024,2024,month=9,['The current place of residence you have list...,"['user_characteristic_country', 'user_connecti...",['The current place of residence you have list...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,NaN
2,2,2,65987,92760,"['Der aktuelle Wohnort, den du in deinem Faceb...",155555220,6.613926e+12,3.654602e+10,wildschytz,Wildschytz,...,9,2024,2024,month=9,['The current place of residence you have spec...,"['user_characteristic_country', 'user_connecti...",['The current place of residence you have spec...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,"['24-65+ years old', 'All', ['Aachen, Germany'..."
3,3,3,58591,144042,"['Información de la conexión, como la direcció...",155336134,1.202113e+17,2.958636e+09,heyjobsofficial,HeyJobs,...,9,2024,2024,month=9,['Connection information such as the IP addres...,"['hashed_list', 'user_characteristic_country',...",['Connection information such as the IP addres...,{'location': ['Connection information such as ...,{'3': ['Connection information such as the IP ...,NaN
4,4,4,68426,144257,"['Der aktuelle Wohnort, den du in deinem Faceb...",155647641,1.202099e+17,3.182336e+08,topazlabs,Topaz Labs,...,9,2024,2024,month=9,['The current place of residence you specified...,"['user_characteristic_country', 'user_connecti...",['The current place of residence you specified...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you spe...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3230,3495,3495,257938,148945,"[""Verbindungsinformationen, wie z. B. die IP-A...",161454174,6.649841e+12,3.171404e+09,drool_art,DROOL,...,2,2025,2025,month=2,"[""Connection information, such as the IP addre...","['user_connection', 'user_activity', 'user_cha...","['Connection information, such as the IP addre...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...","['18-65+ years old', 'All', ['Austria', 'Belgi..."
3231,3496,3496,245674,146766,"[""Der aktuelle Wohnort, den du in deinem Faceb...",161259817,1.202169e+17,8.294187e+09,obsbot,OBSBOT,...,2,2025,2025,month=2,"[""The current place of residence you have list...","['user_characteristic_country', 'user_connecti...",['The current place of residence you have list...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,NaN
3232,3497,3497,295160,147678,[],161852321,6.652784e+12,3.049288e+09,beatsurfing,BEATSURFING,...,2,2025,2025,month=2,NaN,[],NaN,NaN,NaN,"['18-65+ years old', 'All', ['Austria', 'Belgi..."
3233,3498,3498,323588,146671,[],162246556,1.202186e+17,4.782431e+10,openstudiojazz,Open Studio,...,2,2025,2025,month=2,NaN,[],NaN,NaN,NaN,NaN


In [ ]:

# convert english str to list

import re
import json 
import ast
def force_convert_to_list(str_data):
    # str_data = str_data.strip()
    # if str_data.startswith('[') and str_data.endswith(']'):
    #     str_data = str_data[1:-1]
    #print(f"Converting to list: {str_data}")
    try:
       return ast.literal_eval(str_data)  
    except:
        protected = re.sub(r"(\w)'([a-z])", r"\1’", str_data)  # Replace with curly apostrophe
    
        # Step 2: Convert remaining single quotes to double quotes
        standardized = protected.replace("'", '"')

        # Step 3: Restore protected apostrophes
        final_str = standardized.replace("’", "'")

        try:
            return ast.literal_eval(final_str)
        except (SyntaxError, ValueError):
            try:
                # Fallback for malformed strings
                return ast.literal_eval(final_str.replace('"', "'"))
            except:
                return []  # Ultimate fallback


df_insta['english_waist_converted'] = df_insta['english_waist'].apply(
    lambda x: force_convert_to_list(x) if isinstance(x, str) else x
)


In [66]:
# extract age, gender, countries from waist 
# extract age from the ad library
import re

def extract_age_gender_lcoation(value):


    age_gender_coutry_patterns  = {
        'age': [
        r'(aged|ages|age range from) \d+[-+]?\d* to \d+[-+]?\d*',
            r'(an age of|aged|age of|age|age as)? \d+[-+]?\d*( years)? (or|and) (older|over|above)',
            r'(age|aged) between \d+[-+]?\d* and \d+[-+]?\d*',
            r'people (aged|over the age of) \d+',
            r'set their age to \d+',
            r'from the age of',
            r'are between the ages of \d+[-+]?\d* and \d+[-+]?\d*',
            r'over \d+[-+]?\d* years old',
        ],
   
    'gender': [
           r'specified\s+"(\w+)"\s+as their gender',
            r'their gender to',
            r'identified(?: themselves)? as\s+"(\w+)"',
            r'identified(?: themselves)? as\s+“(\w+)”',
            r'(specified|indicated|listed|selected|identified|stated|declared|marked)\s+"(\w+)"\s+as(?: their)? gender',
            r'(specified|indicated|listed|selected|identified|stated|declared|marked)\s+\'(\w+)\'\s+as(?: their)? gender',
            r'(specified|indicated|listed|selected|identified|stated|declared|marked)\s+“(\w+)”\s+as(?: their)? gender',
            r'(specified|indicated|listed|selected|identified|stated|declared|marked)(?: their)? gender as\s+"(\w+)"',
            r'(specified|indicated|listed|selected|identified|stated|declared|marked)(?: their)? gender as\s+“(\w+)”',
            r'(specified|indicated|listed|selected|identified|stated|declared|marked)(?: their)? gender as\s+\'(\w+)\'',
    ], 
    'country' : [
        r'(located primarily|primarily reside|primarily located|mainly located|main location) in ',
        r'based(?: primarily)? in (?:[\w\s]+)',
        r'primary location in (?:[\w\s]+)',
        r'location is',
        r'(?:current|main) (?:place of|place|city|location|residence|town) is',
        r'(?:current|main) (?:place of|place|city|location|residence|town) in',
    ]
    }
    
     ##print(type(value))
    value_lower = value.lower()
    for type_name, regex_list in age_gender_coutry_patterns.items():
        for pattern in regex_list:
            if re.search(pattern, value_lower):
                #print(type_name, value_lower)
                return type_name
    return None
   


def process_targeting_data(row):
    data = row['english_waist_converted']
    classified_items = defaultdict(list)
    result = {}
    if isinstance(data, list):
        for item in data:
            
            targeting_type = extract_age_gender_lcoation(item)
            if targeting_type is None:
                continue
            classified_items[targeting_type].append(item)
            #print(classified_items)
        
        for classification, items in classified_items.items():
            result[classification] = items
                
        return result
    else:     
        return None

df_insta['waist_age_gender_location'] = df_insta.apply(process_targeting_data, axis=1)




In [67]:
df_insta

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,provider_data.user.id,provider_data.waist,provider_data.rawlog_id,advertisement.details.advert_id,advertiser.details.advertiser_id,advertiser.details.advertiser_real_name,advertiser.details.advertiser_internal_name,...,year,country,subfolder,english_waist,targeting_categories_types_german,english_waist_converted,targeting_categories_english,targeting_categories_granularity,selected_ad_library,waist_age_gender_location
0,0,0,75883,133579,"[""Der aktuelle Wohnort, den du in deinem Faceb...",155271982,6.641693e+12,3.203010e+09,unicef_deutschland,UNICEF Deutschland,...,2024,2024,month=9,"[""The current place of residence you indicated...","['user_characteristic_country', 'user_connecti...",[The current place of residence you indicated ...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you ind...,"['18-65+ years old', 'All', ['Germany']]","{'age': ['People aged 18 and over'], 'country'..."
1,1,1,56577,68022,"['Der aktuelle Wohnort, den du in deinem Faceb...",155148521,1.202086e+17,1.530361e+09,lightroom,Adobe Photoshop Lightroom,...,2024,2024,month=9,['The current place of residence you have list...,"['user_characteristic_country', 'user_connecti...",[The current place of residence you have liste...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,NaN,"{'age': ['Persons aged 18 and over'], 'country..."
2,2,2,65987,92760,"['Der aktuelle Wohnort, den du in deinem Faceb...",155555220,6.613926e+12,3.654602e+10,wildschytz,Wildschytz,...,2024,2024,month=9,['The current place of residence you have spec...,"['user_characteristic_country', 'user_connecti...",[The current place of residence you have speci...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,"['24-65+ years old', 'All', ['Aachen, Germany'...","{'age': ['People aged 24 and older'], 'country..."
3,3,3,58591,144042,"['Información de la conexión, como la direcció...",155336134,1.202113e+17,2.958636e+09,heyjobsofficial,HeyJobs,...,2024,2024,month=9,['Connection information such as the IP addres...,"['hashed_list', 'user_characteristic_country',...",[Connection information such as the IP address...,{'location': ['Connection information such as ...,{'3': ['Connection information such as the IP ...,NaN,"{'age': ['Age of 18 years or older'], 'country..."
4,4,4,68426,144257,"['Der aktuelle Wohnort, den du in deinem Faceb...",155647641,1.202099e+17,3.182336e+08,topazlabs,Topaz Labs,...,2024,2024,month=9,['The current place of residence you specified...,"['user_characteristic_country', 'user_connecti...",[The current place of residence you specified ...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you spe...,NaN,"{'age': ['Persons aged 25 and over'], 'country..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3230,3495,3495,257938,148945,"[""Verbindungsinformationen, wie z. B. die IP-A...",161454174,6.649841e+12,3.171404e+09,drool_art,DROOL,...,2025,2025,month=2,"[""Connection information, such as the IP addre...","['user_connection', 'user_activity', 'user_cha...","[Connection information, such as the IP addres...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...","['18-65+ years old', 'All', ['Austria', 'Belgi...","{'age': ['Persons aged 18 and older'], 'countr..."
3231,3496,3496,245674,146766,"[""Der aktuelle Wohnort, den du in deinem Faceb...",161259817,1.202169e+17,8.294187e+09,obsbot,OBSBOT,...,2025,2025,month=2,"[""The current place of residence you have list...","['user_characteristic_country', 'user_connecti...",[The current place of residence you have liste...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,NaN,{'age': ['People who have indicated an age bet...
3232,3497,3497,295160,147678,[],161852321,6.652784e+1

In [82]:
df_insta[df_insta['english_waist_converted'].notnull()]

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,provider_data.user.id,provider_data.waist,provider_data.rawlog_id,advertisement.details.advert_id,advertiser.details.advertiser_id,advertiser.details.advertiser_real_name,advertiser.details.advertiser_internal_name,...,year,country,subfolder,english_waist,targeting_categories_types_german,english_waist_converted,targeting_categories_english,targeting_categories_granularity,selected_ad_library,waist_age_gender_location
0,0,0,75883,133579,"[""Der aktuelle Wohnort, den du in deinem Faceb...",155271982,6.641693e+12,3.203010e+09,unicef_deutschland,UNICEF Deutschland,...,2024,2024,month=9,"[""The current place of residence you indicated...","['user_characteristic_country', 'user_connecti...",[The current place of residence you indicated ...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you ind...,"['18-65+ years old', 'All', ['Germany']]","{'age': ['People aged 18 and over'], 'country'..."
1,1,1,56577,68022,"['Der aktuelle Wohnort, den du in deinem Faceb...",155148521,1.202086e+17,1.530361e+09,lightroom,Adobe Photoshop Lightroom,...,2024,2024,month=9,['The current place of residence you have list...,"['user_characteristic_country', 'user_connecti...",[The current place of residence you have liste...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,NaN,"{'age': ['Persons aged 18 and over'], 'country..."
2,2,2,65987,92760,"['Der aktuelle Wohnort, den du in deinem Faceb...",155555220,6.613926e+12,3.654602e+10,wildschytz,Wildschytz,...,2024,2024,month=9,['The current place of residence you have spec...,"['user_characteristic_country', 'user_connecti...",[The current place of residence you have speci...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,"['24-65+ years old', 'All', ['Aachen, Germany'...","{'age': ['People aged 24 and older'], 'country..."
3,3,3,58591,144042,"['Información de la conexión, como la direcció...",155336134,1.202113e+17,2.958636e+09,heyjobsofficial,HeyJobs,...,2024,2024,month=9,['Connection information such as the IP addres...,"['hashed_list', 'user_characteristic_country',...",[Connection information such as the IP address...,{'location': ['Connection information such as ...,{'3': ['Connection information such as the IP ...,NaN,"{'age': ['Age of 18 years or older'], 'country..."
4,4,4,68426,144257,"['Der aktuelle Wohnort, den du in deinem Faceb...",155647641,1.202099e+17,3.182336e+08,topazlabs,Topaz Labs,...,2024,2024,month=9,['The current place of residence you specified...,"['user_characteristic_country', 'user_connecti...",[The current place of residence you specified ...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you spe...,NaN,"{'age': ['Persons aged 25 and over'], 'country..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3228,3493,3493,329174,148436,"[""Der aktuelle Wohnort, den du in deinem Faceb...",160812259,6.654707e+12,8.088565e+09,getinit_de,get in IT,...,2025,2025,month=2,"[""The current place of residence you have list...","['user_characteristic_country', 'user_connecti...",[The current place of residence you have liste...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,"['18-65+ years old', 'All', ['Germany']]","{'age': ['People aged 18 and over'], 'country'..."
3229,3494,3494,241586,146775,"[""Connection information, such as the IP addre...",161195655,1.202179e+17,2.341008e+07,hm,H&M,...,2025,2025,month=2,"[""Connection information, such as the IP addre...","['user_connection', 'user_characteristic_count...","[Connection information, such as the IP addres...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...",NaN,"{'age': ['An age between 18 and 34'], 'gender'..."
3230,3495,3495,257938,148945,"[""Verbindungsinformationen, wie

In [75]:
# take selected ad library info

df_insat_ad_library = df_insta[df_insta['selected_ad_library'].notnull()]


import re
def extract_age_ranges1(age_text):
    """Extract age ranges from text, handling both 'X to Y' and 'X-Y' formats"""
    pattern = r'(\d+)\s*(?:to|-|and)\s*(\d+)'
    matches = re.findall(pattern, age_text, re.IGNORECASE)

    return [f"{start}-{end}" for start, end in matches]
    
def extract_age_ranges2(age_text):
    """Extract age ranges from text, handling both 'X to Y' and 'X-Y' formats"""
    patterns = [r'(\d+) and over', r'(\d+) and older', r'(\d+) or older', r'(\d+) years and older' ]
    for pattern in patterns:
        matches = re.findall(pattern, age_text, re.IGNORECASE)
        #print(f"Matches found: {matches}")
        if matches != []:
            break
    #print(f"{start}-65+" for start in matches)
    return [f"{start}-65+" for start in matches]
    




def convert_ad_targeting(targeting_dict):
    result = []
    
    # Age conversion
    if 'age' in targeting_dict:
        age_text = targeting_dict['age'][0]
        
        # Extract age ranges
        age_ranges = extract_age_ranges1(age_text)
        if age_ranges:
            result.extend(age_ranges)
        elif '18 and over' in age_text or '18 and older' in age_text:
            result.append('18-65+')
        elif '24 and older' in age_text:
            result.append('24-65+')
        else:
            age_ranges = extract_age_ranges2(age_text)
            result.extend(age_ranges)
    
    # Gender conversion
    if 'gender' in targeting_dict:
        gender_text = targeting_dict['gender'][0]
        if re.search(r'\bmale\b', gender_text, re.IGNORECASE):
            result.append('Men')
        elif re.search(r'\bfemale\b', gender_text, re.IGNORECASE):
            result.append('Women')
    
    # Country conversion - extract country name
    if 'country' in targeting_dict:
        country_text = targeting_dict['country'][0]
        patterns = [
        r'location\s+(?:in|at|within)\s+([A-Za-z\s]+)',
        r'located\s+in\s+([A-Za-z\s]+)',
        r'primary\s+location\s+(?:in|is|at)\s+([A-Za-z\s]+)',
        r'location:\s*([A-Za-z\s]+)',
        r'in\s+([A-Za-z\s]+)$'
    ]
    
        for pattern in patterns:
            match = re.search(pattern, country_text, re.IGNORECASE)
            if match:
                #print(f"Matched country: {match.group(1).strip()}")
                result.append(match.group(1).strip())
                break
    
    return result


df_insat_ad_library['waist_extract'] = df_insat_ad_library['waist_age_gender_location'].apply(
    lambda x: convert_ad_targeting(x) if isinstance(x, dict) else []
) 




/var/folders/ln/nqsht3dd4qb4wzrx8dndsvt8fppz63/T/ipykernel_23752/806567212.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_insat_ad_library['waist_extract'] = df_insat_ad_library['waist_age_gender_location'].apply(


In [76]:
df_insat_ad_library

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,provider_data.user.id,provider_data.waist,provider_data.rawlog_id,advertisement.details.advert_id,advertiser.details.advertiser_id,advertiser.details.advertiser_real_name,advertiser.details.advertiser_internal_name,...,country,subfolder,english_waist,targeting_categories_types_german,english_waist_converted,targeting_categories_english,targeting_categories_granularity,selected_ad_library,waist_age_gender_location,waist_extract
0,0,0,75883,133579,"[""Der aktuelle Wohnort, den du in deinem Faceb...",155271982,6.641693e+12,3.203010e+09,unicef_deutschland,UNICEF Deutschland,...,2024,month=9,"[""The current place of residence you indicated...","['user_characteristic_country', 'user_connecti...",[The current place of residence you indicated ...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you ind...,"['18-65+ years old', 'All', ['Germany']]","{'age': ['People aged 18 and over'], 'country'...","[18-65+, Germany]"
2,2,2,65987,92760,"['Der aktuelle Wohnort, den du in deinem Faceb...",155555220,6.613926e+12,3.654602e+10,wildschytz,Wildschytz,...,2024,month=9,['The current place of residence you have spec...,"['user_characteristic_country', 'user_connecti...",[The current place of residence you have speci...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,"['24-65+ years old', 'All', ['Aachen, Germany'...","{'age': ['People aged 24 and older'], 'country...","[24-65+, Germany]"
11,11,11,70047,134674,"['Connection information, such as the IP addre...",155693574,6.606642e+12,1.108602e+10,tkmaxxat,TK Maxx Österreich,...,2024,month=9,"['Connection information, such as the IP addre...","['user_connection', 'user_characteristic_count...","[Connection information, such as the IP addres...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...","['18-65+ years old', 'All', ['Austria']]","{'age': ['An age of 18 and older'], 'country':...","[18-65+, Austria]"
21,21,21,60766,99143,"['Verbindungsinformationen, wie z. B. die IP-A...",155424335,6.589541e+12,5.549084e+10,hanuta_de,hanuta,...,2024,month=9,"['Connection information, such as the IP addre...","['user_connection', 'user_activity', 'user_cha...","[Connection information, such as the IP addres...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...","['18-39 years old', 'All', ['Germany']]","{'age': ['People aged 18 to 39'], 'country': [...","[18-39, Germany]"
24,24,24,66984,96092,"['Verbindungsinformationen, wie z. B. die IP-A...",155577909,6.652318e+12,1.972219e+09,enlaps,Enlaps,...,2024,month=9,"['Connection information, such as your device'...","['user_connection', 'user_activity', 'user_cha...","[Connection information, such as your device' ...","{'location': [""Connection information, such as...","{'3': [""Connection information, such as your d...","['18-65+ years old', 'All', ['Austria', 'Germa...","{'age': ['People aged 18 and older'], 'country...","[18-65+, Germany]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3216,3481,3481,270093,148366,"[""Verbindungsinformationen, wie z. B. die IP-A...",161517592,6.602446e+12,4.058192e+09,betzold.de,Betzold - Gemeinsam für Bildung,...,2025,month=2,"[""Connection information, such as the IP addre...","['user_connection', 'user_activity', 'user_cha...","[Connection information, such as the IP addres...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...","['18-65+ years old', 'All', ['Austria', 'Germa...","{'age': ['Persons aged 18 and older'], 'countr...","[18-65+, Germany]"
3219,3484,3484,349750,146687,"[""Der aktuelle Wohnort, den du in deinem Faceb...",161102091,6.604520e+12,2.308636e+09,polyversemusic,Polyverse Music,...,2025,month=2,"[""The current place of residence you have list...","['user_characteristic_country', 

In [77]:
# verify inconsistency in the ad library

def verify_consistency(row):
    ad_library_info = row['selected_ad_library']
    waist_info = row['waist_extract']

    for item in waist_info:
        if item not in str(ad_library_info):
            print('problem identified with', item)
            print('ad library info:', ad_library_info)
            print('waist info:', waist_info)
            return False
    return True

df_insat_ad_library.apply(verify_consistency, axis=1)


problem identified with Germany
ad library info: ['18-50 years old', 'All', ['Europe']]
waist info: ['18-50', 'Germany']
problem identified with Germany
ad library info: ['18-65+ years old', 'All', ['Asia', 'Europe', 'South America']]
waist info: ['18-65+', 'Germany']
problem identified with in Germany
ad library info: ['18-65+ years old', 'All', ['Germany']]
waist info: ['18-65+', 'in Germany']
problem identified with Germany
ad library info: ['18-65+ years old', 'All', ['Euro area']]
waist info: ['18-65+', 'Germany']
problem identified with Germany
ad library info: ['18-65+ years old', 'All', ['Euro area']]
waist info: ['18-65+', 'Germany']
problem identified with location is in Germany
ad library info: ['18-65+ years old', 'All', ['Austria', 'Germany']]
waist info: ['18-65+', 'location is in Germany']
problem identified with location is in Germany
ad library info: ['18-65+ years old', 'All', ['Germany']]
waist info: ['18-65+', 'location is in Germany']
problem identified with the Ne

0       True
2       True
11      True
21      True
24      True
        ... 
3216    True
3219    True
3228    True
3230    True
3232    True
Length: 465, dtype: bool

In [80]:
df_insat_ad_library[
    df_insat_ad_library['selected_ad_library'].notnull() &
    df_insat_ad_library['waist_extract'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
]


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,provider_data.user.id,provider_data.waist,provider_data.rawlog_id,advertisement.details.advert_id,advertiser.details.advertiser_id,advertiser.details.advertiser_real_name,advertiser.details.advertiser_internal_name,...,country,subfolder,english_waist,targeting_categories_types_german,english_waist_converted,targeting_categories_english,targeting_categories_granularity,selected_ad_library,waist_age_gender_location,waist_extract
0,0,0,75883,133579,"[""Der aktuelle Wohnort, den du in deinem Faceb...",155271982,6.641693e+12,3.203010e+09,unicef_deutschland,UNICEF Deutschland,...,2024,month=9,"[""The current place of residence you indicated...","['user_characteristic_country', 'user_connecti...",[The current place of residence you indicated ...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you ind...,"['18-65+ years old', 'All', ['Germany']]","{'age': ['People aged 18 and over'], 'country'...","[18-65+, Germany]"
2,2,2,65987,92760,"['Der aktuelle Wohnort, den du in deinem Faceb...",155555220,6.613926e+12,3.654602e+10,wildschytz,Wildschytz,...,2024,month=9,['The current place of residence you have spec...,"['user_characteristic_country', 'user_connecti...",[The current place of residence you have speci...,{'location': ['The current place of residence ...,{'3': ['The current place of residence you hav...,"['24-65+ years old', 'All', ['Aachen, Germany'...","{'age': ['People aged 24 and older'], 'country...","[24-65+, Germany]"
11,11,11,70047,134674,"['Connection information, such as the IP addre...",155693574,6.606642e+12,1.108602e+10,tkmaxxat,TK Maxx Österreich,...,2024,month=9,"['Connection information, such as the IP addre...","['user_connection', 'user_characteristic_count...","[Connection information, such as the IP addres...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...","['18-65+ years old', 'All', ['Austria']]","{'age': ['An age of 18 and older'], 'country':...","[18-65+, Austria]"
21,21,21,60766,99143,"['Verbindungsinformationen, wie z. B. die IP-A...",155424335,6.589541e+12,5.549084e+10,hanuta_de,hanuta,...,2024,month=9,"['Connection information, such as the IP addre...","['user_connection', 'user_activity', 'user_cha...","[Connection information, such as the IP addres...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...","['18-39 years old', 'All', ['Germany']]","{'age': ['People aged 18 to 39'], 'country': [...","[18-39, Germany]"
24,24,24,66984,96092,"['Verbindungsinformationen, wie z. B. die IP-A...",155577909,6.652318e+12,1.972219e+09,enlaps,Enlaps,...,2024,month=9,"['Connection information, such as your device'...","['user_connection', 'user_activity', 'user_cha...","[Connection information, such as your device' ...","{'location': [""Connection information, such as...","{'3': [""Connection information, such as your d...","['18-65+ years old', 'All', ['Austria', 'Germa...","{'age': ['People aged 18 and older'], 'country...","[18-65+, Germany]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3215,3480,3480,344364,93173,"[""Verbindungsinformationen, wie z. B. die IP-A...",161065486,6.681408e+12,1.772075e+09,poloralphlauren,Polo Ralph Lauren,...,2025,month=2,"[""Connection information, such as the IP addre...","['user_connection', 'user_activity', 'user_cha...","[Connection information, such as the IP addres...","{'location': ['Connection information, such as...","{'3': ['Connection information, such as the IP...","['18-45 years old', 'All', ['Germany']]","{'age': ['People aged 18 to 45'], 'country': [...","[18-45, Germany]"
3216,3481,3481,270093,148366,"[""Verbindungsinformationen, wie z. B. die IP-A...",161517592,6.602446e+12,4.058192e+09,betzold.de,Betzold - Gemeinsam für Bildung,...,2025,month=2,"[""Connection information, such as the IP addre...","['user_connection', 'user_activity', 'use